# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and process a real-world biomedical research dataset with the [`mlcroissant`](https://mlcroissant.github.io) library. All dataset elements such as record sets and fields are referenced *by their `@id`*, consistent with best practices for croissant datasets.

### Dataset Source
The dataset source is provided according to a Croissant schema at:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant dataset schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print('Dataset Loaded!\n')
print(f"Title: {metadata.name}\n")
print(f"Description: {metadata.description}\n")
print(f"Version: {metadata.version}")

## 2. Data Overview

Explore the available record sets and their fields. All references are by `@id` as required.

*Note: `mlcroissant` uses the Croissant schema, so record sets and field identifiers follow the structure provided in the metadata.*

In [ ]:
# List all available record sets by @id

record_sets = metadata.record_sets
print(f"Number of record sets: {len(record_sets)}\n")
for i, rs in enumerate(record_sets):
    print(f"{i + 1}. Record set name: {rs.name}")
    print(f"   @id: {rs.id}")
    print(f"   Fields:")
    for field in rs.fields:
        print(f"     - {field.name} (@id: {field.id}, dataType: {getattr(field, 'data_type', 'N/A')})")
    print()

## 3. Data Extraction

Load data from one or more record sets into pandas DataFrames for analysis. All entities use their exact `@id` fields.

*For this example, we use the main clinical record set (`@id` with 'clinical_data' or similar keyword, see listing above).*

In [ ]:
# Let’s extract all available record sets by @id into DataFrames
dataframes = {}
record_set_ids = [rs.id for rs in metadata.record_sets]

for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded record set {rs_id}: shape={df.shape}")
    else:
        print(f"No records found for record set {rs_id}")

# Pick a principal record set for demonstration - use the first one with data
main_record_set_id = next(iter(dataframes.keys()))
print(f"\nMain record set id: {main_record_set_id}")
print(f"Available columns in main record set:")
print(list(dataframes[main_record_set_id].columns))
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)

Process and transform the data for analysis using field `@id`s in all operations.

- **Numeric fields**: Filter, normalize, compute statistics.
- **Group by fields**: Use grouping (e.g., sex, anatomical site). All by `@id`.

Use available fields listed above (see printed columns and `@id`s).

In [ ]:
# Select a numeric field and a group field based on @id (use the field listing above)
# Example: suppose the field IDs (replace if different)
# 'cr:age_at_second_crc_diagnosis' for age at second diagnosis
# 'cr:sex' for biological sex

# Assign field @id variables:
numeric_field = None
group_field = None
for col in dataframes[main_record_set_id].columns:
    if 'age' in col.lower():
        numeric_field = col
    if 'sex' in col.lower() or 'gender' in col.lower():
        group_field = col

print(f"Numeric field: {numeric_field}")
print(f"Group field: {group_field}")

# Filter out records with missing or invalid values
df = dataframes[main_record_set_id]
filtered_df = df.copy()

if numeric_field:
    filtered_df = filtered_df[pd.to_numeric(filtered_df[numeric_field], errors='coerce').notnull()]
    filtered_df[numeric_field] = pd.to_numeric(filtered_df[numeric_field], errors='coerce')

    threshold = filtered_df[numeric_field].mean() # example threshold
    filtered_above_mean = filtered_df[filtered_df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.2f} (n={len(filtered_above_mean)}):")
    display(filtered_above_mean[[numeric_field]].head())

    # Normalize numeric field
    filtered_above_mean[f"{numeric_field}_normalized"] = (
        (filtered_above_mean[numeric_field] - filtered_above_mean[numeric_field].mean()) /
        filtered_above_mean[numeric_field].std()
    )
    print(f"\nNormalized {numeric_field} for filtered records:")
    display(filtered_above_mean[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by group_field if exists
    if group_field and group_field in filtered_above_mean.columns:
        grouped_df = filtered_above_mean.groupby(group_field)[numeric_field].mean().to_frame()
        print(f"\nGrouped data by {group_field} (mean of {numeric_field}):")
        display(grouped_df)
else:
    print("No suitable numeric field found in the record set.")

## 5. Visualization

Visualize the numeric variable's distribution and optionally by grouping field. All axis labels reflect the field `@id`s.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field and numeric_field in df:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=15)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    if group_field and group_field in df:
        plt.figure(figsize=(7,4))
        sns.boxplot(x=df[group_field], y=df[numeric_field])
        plt.title(f"{numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.show()

## 6. Conclusion
This notebook illustrated step-by-step exploration of a Croissant-formatted clinical dataset using only the `@id` fields to refer to record sets and fields throughout:

- Used `mlcroissant` to introspect and load the dataset
- Performed data extraction referencing record sets by their `@id`
- Processed and visualized fields by their `@id`
- Prepared the data (filtering, normalization, grouping) for downstream analysis

This approach is robust for programmatic and reproducible scientific workflows. Refer to the Croissant schema for authoritative documentation of all `@id` fields.